In [1]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import numpy as np
from evedesign.system import System, Protein
from evedesign.models.boltzfold import BoltzFoldTransformer

/Users/khbelahsen/Documents/GitHub/work/marks/evedesign/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[13:47:01] Initializing Normalizer


In [3]:
seq = "TSENPLLALREKISALDEKLLALLAERRELAVEVGKAKLLSHRPVRDIDRERDLLERLITLGKAHHLDAHYITRLFQLIIEDSVLTQQALLQQH"
s = System([Protein(rep=seq, id='EcCM', first_index=2)])
inst = s.rep_to_instance()

In [ ]:
m = BoltzFoldTransformer(device='cpu', use_msa_server=True, diffusion_samples=5)
m.build(s)
output_structures = m.transform([inst])

Processing 1 inputs with 1 threads.


  0%|          | 0/1 [00:00<?, ?it/s]

Generating MSA for /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_pr3tx1lz/inputs/instance_0.yaml with 1 protein entities.
Calling MSA server for target instance_0 with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server


100%|██████████| 1/1 [00:04<00:00,  4.56s/it]
/Users/khbelahsen/Documents/GitHub/work/marks/evedesign/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.0.post0, which is newer than your current Lightning version: v2.5.0
2026-04-13 13:48:20.082 | INFO     | evedesign.models.boltzfold:_load_model:206 - Boltz-2 loaded from /Users/khbelahsen/.boltz/boltz2_conf.ckpt


In [ ]:
result = output_structures[0]
print(f"Score (ptm): {result.score}")
print(f"Confidence (pLDDT): {result.confidence}")

In [ ]:
print("Confidence scores:")
for key, value in result.metadata.items():
    print(f"  {key}: {value}")

# Structure from EntityInstance.models
ei = result[0]
if ei.models:
    chain_id = list(ei.models.keys())[0]
    structure = ei.models[chain_id]
    print(f"\nChain: {chain_id}")
    print(f"Atom count: {len(structure.atom_array)}")
    print(f"Residue range: {structure.atom_array.res_id.min()} - {structure.atom_array.res_id.max()}")
    print(structure.atom_df().head(5))

/Users/khbelahsen/Documents/GitHub/work/marks/evedesign/.venv/lib/python3.12/site-packages/biotite/structure/io/pdbx/convert.py:461: UserWarning: Attribute 'auth_atom_id' not found within 'atom_site' category. The fallback attribute 'label_atom_id' will be used instead
  warnings.warn(
/Users/khbelahsen/Documents/GitHub/work/marks/evedesign/.venv/lib/python3.12/site-packages/biotite/structure/io/pdbx/convert.py:575: UserWarning: Missing 'pdbx_formal_charge' in 'atom_site' category. 'charge' will be set to 0
  warnings.warn(


Confidence scores:
  confidence_score: 0.9245090484619141
  ptm: 0.8647007942199707
  iptm: 0.0
  ligand_iptm: 0.0
  protein_iptm: 0.0
  complex_plddt: 0.9394611716270447
  complex_iplddt: 0.9394611716270447
  complex_pde: 0.34833136200904846
  complex_ipde: 0.0
  chains_ptm: {'0': 0.8647007942199707}
  pair_chains_iptm: {'0': {'0': 0.8647007942199707}}

Chains: ['A']
Atom count: 762


In [ ]:
! pip install py3Dmol -q

In [ ]:
import io
import py3Dmol

ei = result[0]
if ei.models:
    chain_id = list(ei.models.keys())[0]
    structure = ei.models[chain_id]

    buf = io.StringIO()
    structure.to_file(buf, format="cif")
    cif_content = buf.getvalue()

    view = py3Dmol.view(width=800, height=500)
    view.addModel(cif_content, "cif")
    view.setStyle({
        "cartoon": {
            "colorscheme": {
                "prop": "b",
                "gradient": "roygb",
                "min": 50,
                "max": 90
            }
        }
    })
    view.zoomTo()
    view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### Multiple diffusion samples

In [ ]:
# Test: multiple diffusion samples
m_multi = BoltzFoldTransformer(
    device='cpu',
    use_msa_server=True,
    diffusion_samples=3,
)
m_multi.build(s)
predictions_dir_multi, files_multi = m_multi.transform([inst])

cif_multi = [f for f in files_multi if f.suffix == ".cif"]
json_multi = [f for f in files_multi if f.suffix == ".json"]
print(f"CIF files ({len(cif_multi)}):")
for f in cif_multi:
    print(f"  {f.name}")
print(f"JSON files ({len(json_multi)}):")
for f in json_multi:
    print(f"  {f.name}")

Processing 1 inputs with 1 threads.


  0%|          | 0/1 [00:00<?, ?it/s]

Generating MSA for /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_bs47eajg/inputs/instance_0.yaml with 1 protein entities.
Calling MSA server for target instance_0 with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server


100%|██████████| 1/1 [00:04<00:00,  4.42s/it]
/Users/khbelahsen/Documents/GitHub/work/marks/evedesign/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.0.post0, which is newer than your current Lightning version: v2.5.0
2026-04-10 12:49:57.288 | INFO     | evedesign.models.boltzfold:_load_model:205 - Boltz-2 loaded from /Users/khbelahsen/.boltz/boltz2_conf.ckpt
2026-04-10 13:00:24.980 | INFO     | evedesign.models.boltzfold:transform:360 - Boltz-2 output written to: /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_bs47eajg/predictions
2026-04-10 13:00:24.995 | INFO     | evedesign.models.boltzfold:transform:361 - Files written (16):
2026-04-10 13:00:24.998 | INFO     | evedesign.models.boltzfold:transform:364 -   instance_0/confidence_instance_0_model_0.json (442 bytes)
2026-04-10 13:00:24.999 | INFO     | evedesign.models.boltzfold:transform:364 -   instance_0/confidence_instance_0_mod

CIF files (3):
  instance_0_model_0.cif
  instance_0_model_1.cif
  instance_0_model_2.cif
JSON files (3):
  confidence_instance_0_model_0.json
  confidence_instance_0_model_1.json
  confidence_instance_0_model_2.json


In [25]:
files_multi

[PosixPath('/var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_bs47eajg/predictions/instance_0/confidence_instance_0_model_0.json'),
 PosixPath('/var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_bs47eajg/predictions/instance_0/confidence_instance_0_model_1.json'),
 PosixPath('/var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_bs47eajg/predictions/instance_0/confidence_instance_0_model_2.json'),
 PosixPath('/var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_bs47eajg/predictions/instance_0/instance_0_model_0.cif'),
 PosixPath('/var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_bs47eajg/predictions/instance_0/instance_0_model_1.cif'),
 PosixPath('/var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_bs47eajg/predictions/instance_0/instance_0_model_2.cif'),
 PosixPath('/var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_bs47eajg/predictions/instance_0/pae_instance_0_model_0.npz'),
 PosixPath('/var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000g

In [ ]:
if cif_files:
    for cif_file in cif_multi:
        view = py3Dmol.view(width=200, height=200)
        view.addModel(cif_file.read_text(), "cif")
        view.setStyle({
            "cartoon": {
                "colorscheme": {
                    "prop": "b",
                    "gradient": "roygb",
                    "min": 50,
                    "max": 90
                }
            }
        })
        view.zoomTo()
        view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [21]:
import numpy as np
from evedesign.system import System, Protein, SystemInstance, EntityInstance
from evedesign.models.boltzfold import BoltzFoldTransformer

wt = "TSENPLLALREKISALDEKLLALLAERRELAVEVGKAKLLSHRPVRDIDRERDLLERLITLGKAHHLDAHYITRLFQLIIEDSVLTQQALLQQH"

# Bind the model to the WT system once
s = System([Protein(rep=wt, id='EcCM', first_index=2)])
m = BoltzFoldTransformer(device='cpu', use_msa_server=True).build(s)

# Generate all single-point mutants at, say, position 10 (first_index=2 → array idx 8)
AA = "ACDEFGHIKLMNPQRSTVWY"
variants = []
pos = 8  # array index in rep
for aa in AA:
    if aa == wt[pos]:
        continue
    mut = wt[:pos] + aa + wt[pos+1:]
    variants.append(mut)

# One SystemInstance per variant — all length 94, all match the bound system
instances = [
    SystemInstance([EntityInstance(rep=np.array(list(v), dtype='U1'))])
    for v in variants
]

predictions_dir, files = m.transform(instances)


Processing 19 inputs with 1 threads.


  0%|          | 0/19 [00:00<?, ?it/s]

Generating MSA for /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_sp_uf9w8/inputs/instance_0.yaml with 1 protein entities.
Calling MSA server for target instance_0 with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server


Sleeping for 6s. Reason: PENDING
Sleeping for 8s. Reason: RUNNING
  5%|▌         | 1/19 [00:19<05:56, 19.79s/it]

Generating MSA for /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_sp_uf9w8/inputs/instance_1.yaml with 1 protein entities.
Calling MSA server for target instance_1 with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server


...

Sleeping for 8s. Reason: PENDING
 89%|████████▉ | 17/19 [04:16<00:27, 13.82s/it]

Generating MSA for /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_sp_uf9w8/inputs/instance_17.yaml with 1 protein entities.
Calling MSA server for target instance_17 with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server


Sleeping for 8s. Reason: PENDING
 95%|█████████▍| 18/19 [04:29<00:13, 13.45s/it]

Generating MSA for /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_sp_uf9w8/inputs/instance_18.yaml with 1 protein entities.
Calling MSA server for target instance_18 with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server


Sleeping for 7s. Reason: PENDING
100%|██████████| 19/19 [04:41<00:00, 14.79s/it]
/Users/khbelahsen/Documents/GitHub/work/marks/evedesign/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.0.post0, which is newer than your current Lightning version: v2.5.0
2026-04-10 13:21:40.906 | INFO     | evedesign.models.boltzfold:_load_model:205 - Boltz-2 loaded from /Users/khbelahsen/.boltz/boltz2_conf.ckpt
2026-04-10 16:56:46.725 | INFO     | evedesign.models.boltzfold:transform:360 - Boltz-2 output written to: /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_sp_uf9w8/predictions
2026-04-10 16:56:46.759 | INFO     | evedesign.models.boltzfold:transform:361 - Files written (114):
2026-04-10 16:56:46.766 | INFO     | evedesign.models.boltzfold:transform:364 -   instance_0/confidence_instance_0_model_0.json (443 bytes)
2026-04-10 16:56:46.767 | INFO     | evedesign.models.boltzfold:transform:364 -   

### Test (a): single protein, single instance

In [ ]:
s_a = System([Protein(rep='MAST', id='test_a')])
results_a = BoltzFoldTransformer(
    device='cpu', use_msa=False,
    sampling_steps=1, recycling_steps=1
).build(s_a).transform([s_a.rep_to_instance()])
ei = results_a[0][0]
assert ei.models is not None, "models is None"
assert results_a[0].score is not None, "score is None"

### Test (b): multiple instances

In [ ]:
s_b = System([Protein(rep='MAST', id='test_b')])
inst_b1 = s_b.rep_to_instance()
inst_b2 = s_b.rep_to_instance()
results_b = BoltzFoldTransformer(
    device='cpu', use_msa=False,
).build(s_b).transform([inst_b1, inst_b2])
assert len(results_b) == 2, f"expected 2 results got {len(results_b)}"
assert results_b[0][0].models is not None
assert results_b[1][0].models is not None


### Test (c): two-chain complex

In [ ]:
s_c = System([
    Protein(rep='MAST', id='chain1'),
    Protein(rep='GKLT', id='chain2'),
])
results_c = BoltzFoldTransformer(
    device='cpu', use_msa=False,
).build(s_c).transform([s_c.rep_to_instance()])
ei0 = results_c[0][0]
ei1 = results_c[0][1]
assert ei0.models is not None, "entity 0 models is None"
assert ei1.models is not None, "entity 1 models is None"
assert list(ei0.models.keys()) == ["A"], \
    f"expected chain A got {list(ei0.models.keys())}"
assert list(ei1.models.keys()) == ["B"], \
    f"expected chain B got {list(ei1.models.keys())}"

### Test (d): entity parameter

In [ ]:
s_d = System([Protein(rep='MAST', id='test_d')])
m_d = BoltzFoldTransformer(
    device='cpu', use_msa=False,
    sampling_steps=1, recycling_steps=1
).build(s_d)
try:
    m_d.transform([s_d.rep_to_instance()], entity=1)
    print("FAIL: expected NotImplementedError")
except NotImplementedError:
    print("PASS: NotImplementedError raised as expected")

###  Test (e): Multiple diffusion_samples

In [ ]:
s_e = System([Protein(rep='MAST', id='test_e')])
results_e = BoltzFoldTransformer(
    device='cpu', use_msa=False,
    sampling_steps=1, recycling_steps=1,
    diffusion_samples=3,
).build(s_e).transform([s_e.rep_to_instance()])
ei = results_e[0][0]
assert ei.models is not None, "models is None"
assert results_e[0].score is not None, "score is None"
print(f"PASS: got structure from best of 3 samples")
print(f"  score: {results_e[0].score}")



In [ ]:
print("=== Test (f): homo-oligomer copies=2 ===")
try:
    s_f = System([Protein(rep='MAST', id='test_f', copies=2)])
    results_f = BoltzFoldTransformer(
        device='mps', use_msa=False,
        sampling_steps=1, recycling_steps=1,
    ).build(s_f).transform([s_f.rep_to_instance()])
    ei = results_f[0][0]
    assert ei.models is not None, "models is None"
    chains = list(ei.models.keys())
    assert "A" in chains, f"chain A missing, got {chains}"
    assert "B" in chains, f"chain B missing, got {chains}"
    print(f"PASS: both chains present: {chains}")
except Exception as e:
    print(f"FAIL: {e}")


In [ ]:
print("=== Test (g): entity with MSA sequences ===")
try:
    from evedesign.sequence import Sequences, Sequence
    seqs = Sequences([
        Sequence(seq='MAST', id='hom1'),
        Sequence(seq='VAST', id='hom2'),
    ])
    s_g = System([
        Protein(rep='MAST', id='test_g', sequences=seqs)
    ])
    results_g = BoltzFoldTransformer(
        device='mps', use_msa=True,
        sampling_steps=1, recycling_steps=1,
    ).build(s_g).transform([s_g.rep_to_instance()])
    ei = results_g[0][0]
    assert ei.models is not None, "models is None"
    print("PASS: folded with MSA sequences")
except Exception as e:
    print(f"FAIL: {e}")


In [ ]:
print("=== Test (h): entity with structures attached ===")
import warnings
try:
    from evedesign.structure import StructureFile
    ei_prev = result[0]
    dummy_structure = result[0].models.get("A") if result[0].models else None
    if dummy_structure is None:
        print("SKIP: no structure available from previous cells")
    else:
        s_h = System([Protein(
            rep=seq, id='test_h', first_index=2,
            structures={"model1": dummy_structure}
        )])
        results_h = BoltzFoldTransformer(
            device='mps', use_msa=False,
            sampling_steps=1, recycling_steps=1,
        ).build(s_h).transform([s_h.rep_to_instance()])
        assert results_h[0][0].models is not None
        print("PASS: ran successfully, structures ignored with warning")
except Exception as e:
    print(f"FAIL: {e}")


In [ ]:
print("=== Test (i): non-protein entity ===")
try:
    from evedesign.system import DNA
    s_i = System([DNA(rep='ATCG', id='test_i')])
    m_i = BoltzFoldTransformer(device='mps', use_msa=False)
    ok, msg = m_i.can_model(s_i)
    assert not ok, "can_model should return False for DNA"
    try:
        m_i.build(s_i)
        print("FAIL: expected ValueError from build()")
    except ValueError as e:
        print(f"PASS: correctly rejected with: {e}")
except Exception as e:
    print(f"FAIL: unexpected error: {e}")
